# Análisis de Métricas Estructurales

Este notebook describe en detalle las **5 métricas estructurales** utilizadas en NeuralPlumber  
para evaluar la calidad de los niveles generados por el DCGAN.  

**Contenido:**
1. Setup e imports
2. Descripción teórica de cada métrica
3. Cálculo paso a paso sobre un nivel del baseline (Exp A)
4. Comparación visual y numérica: nivel baseline vs. nivel CMA-ES (Exp C)
5. Distribución de métricas sobre los 40 runs de CMA-ES
6. Análisis: qué métrica mejoró más y por qué

## 1. Setup — Imports y configuración de paths

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Paths relativos desde notebooks/ ──────────────────────────────────────────
NOTEBOOK_DIR    = os.path.abspath('')
PROJECT_ROOT    = os.path.abspath('../../')
DAGSTUHL_PATH   = os.path.join(PROJECT_ROOT, 'clone', 'DagstuhlGAN', 'pytorch')
SRC_PATH        = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'src')
BASELINE_DIR    = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'baseline')
SF_DIR          = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'structural_fitness')

for p in [DAGSTUHL_PATH, SRC_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Python    :', sys.version)
print('DAGSTUHL  :', os.path.isdir(DAGSTUHL_PATH))
print('SRC_PATH  :', os.path.isdir(SRC_PATH))
print('BASELINE  :', os.path.isdir(BASELINE_DIR))
print('SF_DIR    :', os.path.isdir(SF_DIR))

## 2. Las 5 Métricas Estructurales

Todas las métricas están implementadas en `src/metrics/structural.py` y son calculadas  
por la función `structural_score(level)`, que recibe un `np.ndarray` de forma `(14, 28)`  
con valores enteros 0–9 (IDs de tiles) y devuelve un diccionario con las 5 métricas.

### Convención de tiles

| ID | Nombre | Descripción |
|---|---|---|
| 0 | Suelo | Bloque sólido principal |
| 1 | Breakable | Bloque rompible |
| 2 | Vacío | Espacio abierto / cielo |
| 3 | Q-lleno | Bloque pregunta con ítem |
| 4 | Q-vacío | Bloque pregunta vacío |
| 5 | Enemigo | Goomba u otro enemigo |
| 6 | Pipe TL | Tubería arriba-izquierda |
| 7 | Pipe TR | Tubería arriba-derecha |
| 8 | Pipe L | Tubería cuerpo-izquierda |
| 9 | Pipe R | Tubería cuerpo-derecha |

---

### Métrica 1 — `pipe_completeness` (Completitud de tuberías)

**Qué mide:** Fracción de tuberías que son estructuralmente válidas.  
Una tubería válida en Mario tiene la forma:
```
[ 6 | 7 ]   ← fila superior (TL, TR)
[ 8 | 9 ]   ← fila(s) de cuerpo (L, R), al menos una
```
**Cómo se calcula:** Se buscan pares `(TL, TR)` horizontalmente adyacentes.  
Para cada par encontrado, se verifica que directamente debajo haya al menos una fila `(L, R)`.  
La métrica es `tuberías_válidas / max(tuberías_candidatas, 1)`.

**Rango:** [0, 1]  
**Qué indica:**  
- `0.0` → No hay ninguna tubería estructuralmente completa  
- `1.0` → Todas las tuberías encontradas tienen cabeza y cuerpo correctos  
- **Baseline Exp A: 0.2075** — muy bajo, el generador rara vez produce tuberías válidas

---

### Métrica 2 — `ground_continuity` (Continuidad del suelo)

**Qué mide:** Proporción de columnas en las que la fila inferior (fila 13) es suelo sólido.  
**Cómo se calcula:** Se cuenta cuántas columnas de las 28 tienen tile sólido (IDs 0 o 1)  
en la última fila. La métrica es `columnas_con_suelo / 28`.

**Rango:** [0, 1]  
**Qué indica:**  
- `0.0` → El suelo está completamente ausente (nivel imposible de jugar)  
- `1.0` → Suelo continuo en todas las columnas  
- **Baseline Exp A: 0.5093** — solo ~14 de 28 columnas tienen suelo sólido

---

### Métrica 3 — `gap_traversability` (Traversabilidad de huecos)

**Qué mide:** Fracción de huecos (columnas sin suelo) que son lo suficientemente  
estrechos para ser saltados por Mario.  
**Cómo se calcula:** Se identifican secuencias consecutivas de columnas sin suelo (gaps).  
Un gap es "traversable" si su ancho es ≤ 4 tiles (límite de salto de Mario).  
La métrica es `gaps_traversables / max(total_gaps, 1)`. Si no hay gaps, vale 1.0.

**Rango:** [0, 1]  
**Qué indica:**  
- `0.0` → Todos los huecos son demasiado anchos para saltarlos  
- `1.0` → Todos los huecos son salvables (o no hay huecos)  
- **Baseline Exp A: 0.9100** — ya era alto, el generador produce huecos estrechos

---

### Métrica 4 — `enemy_placement` (Colocación de enemigos)

**Qué mide:** Fracción de enemigos que están correctamente colocados sobre suelo sólido.  
**Cómo se calcula:** Para cada tile de tipo Enemigo (ID 5), se verifica que el tile  
directamente debajo (fila + 1) sea sólido (ID 0 o 1).  
La métrica es `enemigos_bien_colocados / max(total_enemigos, 1)`. Sin enemigos → 1.0.

**Rango:** [0, 1]  
**Qué indica:**  
- `0.0` → Todos los enemigos flotan en el aire  
- `1.0` → Todos los enemigos están sobre suelo (o no hay enemigos)  
- **Baseline Exp A: 0.9667** — el generador aprendió bien este patrón local

---

### Métrica 5 — `structural_avg` (Promedio estructural)

**Qué mide:** Media aritmética de las 4 métricas anteriores.  
**Cómo se calcula:**  
```python
structural_avg = (pipe_completeness + ground_continuity +
                  gap_traversability + enemy_placement) / 4
```
**Rango:** [0, 1]  
**Qué indica:** Calidad estructural global del nivel. Umbral objetivo: **0.70**.  
- **Baseline Exp A: 0.6484** — por debajo del umbral  
- **Exp C (CMA-ES): 0.8223** — supera claramente el umbral

## 3. Cálculo paso a paso sobre un nivel del Baseline (Exp A)

In [ ]:
# ── Cargar niveles del baseline ────────────────────────────────────────────────
sample_path = os.path.join(BASELINE_DIR, 'sample_levels.json')

with open(sample_path, 'r') as f:
    sample_levels_raw = json.load(f)

# Convertir a np.ndarray (14, 28)
sample_levels_a = [np.array(lv, dtype=np.int32) for lv in sample_levels_raw]

print(f'Niveles cargados: {len(sample_levels_a)}')
print(f'Forma de cada nivel: {sample_levels_a[0].shape}')
print(f'IDs de tile presentes: {np.unique(sample_levels_a[0])}')

In [ ]:
from visualization.level_renderer import render_level

# Usar el primer nivel del baseline
nivel_a = sample_levels_a[0]

print('Nivel baseline seleccionado (IDs de tile):')
print(nivel_a)
print(f'\nForma: {nivel_a.shape}  |  Valores únicos: {np.unique(nivel_a)}')

render_level(nivel_a, title='Nivel baseline — Exp A (sample_levels[0])')

In [ ]:
from metrics.structural import structural_score

# ── Calcular métricas del nivel baseline ──────────────────────────────────────
metrics_a = structural_score(nivel_a)

METRIC_LABELS = {
    'pipe_completeness' : 'Completitud de tuberías',
    'ground_continuity' : 'Continuidad de suelo',
    'gap_traversability': 'Traversabilidad de gaps',
    'enemy_placement'   : 'Colocación de enemigos',
    'structural_avg'    : 'Promedio estructural',
}

GOOD_THRESHOLD = 0.70

print('Métricas del nivel baseline (Exp A):')
print('=' * 60)
for key, label in METRIC_LABELS.items():
    val    = metrics_a[key]
    estado = 'OK  ' if val >= GOOD_THRESHOLD else 'BAJO'
    print(f'  {label:<33} {val:.4f}  [{estado}]')
print('=' * 60)

# ── Análisis paso a paso: pipe_completeness ───────────────────────────────────
print('\n--- Desglose de pipe_completeness ---')
pipe_heads = []
for row in range(nivel_a.shape[0] - 1):
    for col in range(nivel_a.shape[1] - 1):
        if nivel_a[row, col] == 6 and nivel_a[row, col + 1] == 7:
            has_body = (nivel_a[row + 1, col] == 8 and nivel_a[row + 1, col + 1] == 9)
            pipe_heads.append({'pos': (row, col), 'valid': has_body})

print(f'  Pares TL-TR encontrados: {len(pipe_heads)}')
for ph in pipe_heads:
    status = 'COMPLETA' if ph['valid'] else 'incompleta (sin cuerpo)'
    print(f'    Fila {ph["pos"][0]}, Col {ph["pos"][1]}: {status}')

# ── Análisis paso a paso: ground_continuity ───────────────────────────────────
print('\n--- Desglose de ground_continuity ---')
ground_row   = nivel_a[-1, :]          # fila 13
solid_cols   = np.sum((ground_row == 0) | (ground_row == 1))
print(f'  Fila 13 (suelo): {ground_row.tolist()}')
print(f'  Columnas con suelo sólido: {solid_cols} / 28  = {solid_cols/28:.4f}')

# ── Análisis paso a paso: enemy_placement ────────────────────────────────────
print('\n--- Desglose de enemy_placement ---')
enemy_positions = list(zip(*np.where(nivel_a == 5)))
print(f'  Enemigos encontrados: {len(enemy_positions)}')
for (r, c) in enemy_positions:
    if r + 1 < nivel_a.shape[0]:
        below = nivel_a[r + 1, c]
        ok    = (below == 0 or below == 1)
        print(f'    Enemigo en ({r},{c}), tile debajo = {below}  -> {"OK" if ok else "FLOTANDO"}')
    else:
        print(f'    Enemigo en fila {r} (borde inferior) -> fuera de rango')

In [ ]:
# ── Gráfico de barras horizontales ────────────────────────────────────────────
keys   = [k for k in METRIC_LABELS if k != 'structural_avg']
labels = [METRIC_LABELS[k] for k in keys]
values = [metrics_a[k] for k in keys]
colors = ['#e74c3c' if v < GOOD_THRESHOLD else '#2ecc71' for v in values]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, values, color=colors, edgecolor='#333', linewidth=0.7)
ax.axvline(GOOD_THRESHOLD, color='#555', linestyle='--', linewidth=1.2,
           label=f'Umbral objetivo ({GOOD_THRESHOLD})')
ax.set_xlim(0, 1.10)
ax.set_xlabel('Score (0 – 1)')
ax.set_title('Métricas del nivel baseline — Exp A (sample_levels[0])',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')

for bar, val in zip(bars, values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

ok_patch   = mpatches.Patch(color='#2ecc71', label='Supera umbral')
bajo_patch = mpatches.Patch(color='#e74c3c', label='Por debajo del umbral')
ax.legend(handles=[ok_patch, bajo_patch,
           plt.Line2D([0],[0], color='#555', linestyle='--', linewidth=1.2,
                      label=f'Umbral ({GOOD_THRESHOLD})')],
          loc='lower right')

plt.tight_layout()
plt.show()

print(f'\nPromedio estructural del nivel: {metrics_a["structural_avg"]:.4f}')

## 4. Comparación: Nivel Baseline vs. Nivel CMA-ES

In [ ]:
# ── Cargar el mejor nivel del CMA-ES ──────────────────────────────────────────
best_path = os.path.join(SF_DIR, 'best_levels_netG_15levels_f3_static.json')

with open(best_path, 'r') as f:
    best_levels_raw = json.load(f)

best_levels_c = [np.array(lv, dtype=np.int32) for lv in best_levels_raw]

print(f'Niveles CMA-ES cargados: {len(best_levels_c)}')
print(f'Forma de cada nivel: {best_levels_c[0].shape}')

# Usar el primer nivel del CMA-ES
nivel_c = best_levels_c[0]

In [ ]:
from visualization.level_renderer import render_comparison

# ── Comparación visual lado a lado ────────────────────────────────────────────
render_comparison(
    [nivel_a, nivel_c],
    titles=[
        'Nivel baseline — Exp A (DCGAN aleatorio)',
        'Nivel optimizado — Exp C (CMA-ES + F3)'
    ]
)

In [ ]:
# ── Calcular métricas de ambos niveles ────────────────────────────────────────
metrics_c = structural_score(nivel_c)

print('Comparación de métricas: nivel baseline vs. nivel CMA-ES')
print('=' * 72)
print(f"{'Métrica':<33} {'Exp A':>8}  {'Exp C':>8}  {'Δ':>8}  {'%Mejora':>8}")
print('=' * 72)

for key, label in METRIC_LABELS.items():
    va     = metrics_a[key]
    vc     = metrics_c[key]
    delta  = vc - va
    pct    = (delta / va * 100) if va > 0 else float('nan')
    arrow  = '↑' if delta > 0.001 else ('↓' if delta < -0.001 else '=')
    print(f'  {label:<31} {va:>8.4f}  {vc:>8.4f}  {delta:>+8.4f}  {arrow}{pct:>+6.1f}%')

print('=' * 72)

# ── Gráfico comparativo ───────────────────────────────────────────────────────
metric_keys  = list(METRIC_LABELS.keys())
metric_short = ['pipe\ncompleteness', 'ground\ncontinuity', 'gap\ntraversability',
                'enemy\nplacement', 'structural\navg']

vals_a = [metrics_a[k] for k in metric_keys]
vals_c = [metrics_c[k] for k in metric_keys]

x = np.arange(len(metric_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars_a = ax.bar(x - w/2, vals_a, w, label='Exp A — Baseline (DCGAN)',
                color='#3498db', edgecolor='#222', linewidth=0.7)
bars_c = ax.bar(x + w/2, vals_c, w, label='Exp C — CMA-ES optimizado',
                color='#e74c3c', edgecolor='#222', linewidth=0.7)

ax.axhline(GOOD_THRESHOLD, color='#888', linestyle='--', linewidth=1.2,
           label=f'Umbral objetivo ({GOOD_THRESHOLD})')
ax.set_xticks(x)
ax.set_xticklabels(metric_short, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score (0 – 1)')
ax.set_title('Comparación de métricas: un nivel Exp A vs. un nivel Exp C',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

for bar, val in zip(bars_a, vals_a):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, color='#1a5276')
for bar, val in zip(bars_c, vals_c):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, color='#922b21')

plt.tight_layout()
plt.show()

## 5. Distribución de métricas sobre los 40 runs de CMA-ES

In [ ]:
# ── Cargar resultados completos de CMA-ES ─────────────────────────────────────
results_path = os.path.join(SF_DIR, 'cmaes_results_netG_15levels_f3_static.json')

with open(results_path, 'r') as f:
    cmaes_data = json.load(f)

n_runs      = cmaes_data['n_runs']
avg_metrics = cmaes_data['avg_metrics']
runs        = cmaes_data['runs']

print(f'Número de runs CMA-ES: {n_runs}')
print(f'Métricas promedio:')
for key, label in METRIC_LABELS.items():
    print(f'  {label:<33}: {avg_metrics[key]:.4f}')
print(f'\nEjemplo de un run:')
r0 = runs[0]
print(f'  Run {r0["run"]} | best_score={r0["best_score"]:.4f} | n_evals={r0["n_evals"]} | metrics={r0["metrics"]}')

In [ ]:
# ── Extraer métricas por run ───────────────────────────────────────────────────
metric_keys_base = ['pipe_completeness', 'ground_continuity',
                    'gap_traversability', 'enemy_placement', 'structural_avg']

per_run = {k: [] for k in metric_keys_base}

for run in runs:
    m = run['metrics']
    for k in metric_keys_base:
        per_run[k].append(m[k])

# Estadísticas descriptivas
print(f'Estadísticas descriptivas sobre {n_runs} runs CMA-ES:')
print('=' * 70)
print(f"{'Métrica':<25} {'Min':>7} {'Q1':>7} {'Med':>7} {'Q3':>7} {'Max':>7} {'Mean':>7}")
print('=' * 70)

short_names = {
    'pipe_completeness' : 'pipe_completeness',
    'ground_continuity' : 'ground_continuity',
    'gap_traversability': 'gap_traversability',
    'enemy_placement'   : 'enemy_placement',
    'structural_avg'    : 'structural_avg',
}

for k in metric_keys_base:
    arr = np.array(per_run[k])
    print(f'  {short_names[k]:<23} '
          f'{arr.min():>7.4f} '
          f'{np.percentile(arr,25):>7.4f} '
          f'{np.median(arr):>7.4f} '
          f'{np.percentile(arr,75):>7.4f} '
          f'{arr.max():>7.4f} '
          f'{arr.mean():>7.4f}')

print('=' * 70)

In [ ]:
# ── Violin plot de las 5 métricas ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 5), sharey=True)

colors_v = ['#9b59b6', '#2980b9', '#27ae60', '#e67e22', '#e74c3c']
short_labels = [
    'pipe\ncompleteness',
    'ground\ncontinuity',
    'gap\ntraversability',
    'enemy\nplacement',
    'structural\navg'
]

for i, (k, col, label) in enumerate(zip(metric_keys_base, colors_v, short_labels)):
    data_k = per_run[k]
    arr    = np.array(data_k)

    parts = axes[i].violinplot(data_k, positions=[0], showmedians=True,
                                showextrema=True)

    for pc in parts['bodies']:
        pc.set_facecolor(col)
        pc.set_alpha(0.7)
    for part_name in ['cbars', 'cmins', 'cmaxes', 'cmedians']:
        if part_name in parts:
            parts[part_name].set_color('#222')
            parts[part_name].set_linewidth(1.5)

    # Puntos individuales (jitter)
    jitter = np.random.default_rng(seed=i).uniform(-0.08, 0.08, len(data_k))
    axes[i].scatter(jitter, data_k, s=15, color='#333', alpha=0.5, zorder=5)

    # Línea de umbral
    axes[i].axhline(GOOD_THRESHOLD, color='red', linestyle='--',
                    linewidth=1.0, alpha=0.8)

    # Media
    axes[i].axhline(arr.mean(), color=col, linestyle='-',
                    linewidth=2.0, alpha=0.9, label=f'Media: {arr.mean():.3f}')

    axes[i].set_title(label, fontsize=9, fontweight='bold')
    axes[i].set_xticks([])
    axes[i].set_ylim(-0.05, 1.08)
    axes[i].legend(loc='lower right', fontsize=7)
    axes[i].grid(axis='y', linestyle='--', alpha=0.3)

axes[0].set_ylabel('Score (0 – 1)')
fig.suptitle(f'Distribución de métricas estructurales — {n_runs} runs CMA-ES',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplot complementario (todas las métricas en un solo eje) ────────────────
fig, ax = plt.subplots(figsize=(12, 5))

data_all   = [per_run[k] for k in metric_keys_base]
bp = ax.boxplot(data_all, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))

colors_bp = ['#9b59b6', '#2980b9', '#27ae60', '#e67e22', '#e74c3c']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(GOOD_THRESHOLD, color='gray', linestyle='--', linewidth=1.5,
           label=f'Umbral objetivo ({GOOD_THRESHOLD})')

# Mostrar media
for j, k in enumerate(metric_keys_base):
    mean_val = np.mean(per_run[k])
    ax.plot(j + 1, mean_val, 'D', color='white', markersize=7,
            markeredgecolor='black', zorder=10)
    ax.text(j + 1, mean_val + 0.02, f'{mean_val:.3f}', ha='center',
            fontsize=8, color='#222')

ax.set_xticks(range(1, 6))
ax.set_xticklabels(short_labels, fontsize=9)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score (0 – 1)')
ax.set_title(f'Boxplot de métricas estructurales — {n_runs} runs CMA-ES  (◆ = media)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower left')
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 6. Análisis: ¿Qué métrica mejoró más y por qué?

### Tabla resumen: Exp A (baseline) vs. Exp C (CMA-ES)

| Métrica | Exp A (baseline) | Exp C (CMA-ES avg) | Δ absoluto | % mejora |
|---|---|---|---|---|
| `pipe_completeness`  | 0.2075 | **0.4567** | +0.2492 | **+120%** |
| `ground_continuity`  | 0.5093 | **0.8326** | +0.3233 | +64% |
| `gap_traversability` | 0.9100 | **1.0000** | +0.0900 | +10% |
| `enemy_placement`    | 0.9667 | **1.0000** | +0.0333 | +3% |
| `structural_avg`     | 0.6484 | **0.8223** | +0.1739 | **+27%** |

### La métrica que más mejoró: `pipe_completeness` (+120%)

**Por qué mejoró tanto:**

1. **La función F3 penaliza explícitamente las tuberías incompletas.**  
   La función de fitness `f3_static` incluye `pipe_completeness` como componente directa  
   de su función objetivo. El CMA-ES navega el espacio latente buscando vectores `z` que  
   maximicen esta métrica junto con las demás.

2. **`pipe_completeness` parte de un nivel muy bajo (0.2075)**, lo que deja mucho margen  
   de mejora relativa. Las métricas ya altas (`enemy_placement` = 0.9667, `gap_traversability`  
   = 0.91) tienen poco espacio para mejorar.

3. **Las tuberías son estructuras de alta coherencia local**: requieren exactamente 4 tiles  
   adyacentes en posiciones relativas fijas (TL, TR en la cabeza; L, R en el cuerpo).  
   El generador DCGAN tiene esta capacidad latente —el espacio latente contiene regiones  
   donde el generador sí produce tuberías válidas—, pero el muestreo aleatorio raramente  
   las alcanza. CMA-ES *busca* estas regiones sistemáticamente.

4. **`ground_continuity` mejoró en segundo lugar (+64%)** porque el suelo es también  
   una estructura espacialmente extendida (involucra múltiples columnas consecutivas),  
   y la función F3 la considera directamente.

### El efecto del espacio latente suave

El hecho de que CMA-ES converja en ~20 generaciones (observado en el notebook 04)  
es evidencia de que el espacio latente del DCGAN tiene **gradiente suave** respecto a  
la calidad estructural: pequeños cambios en `z` producen cambios graduales en las  
métricas, lo que permite que CMA-ES (un algoritmo de búsqueda sin gradiente) encuentre  
soluciones buenas con relativamente pocas evaluaciones.

### Por qué `gap_traversability` y `enemy_placement` ya eran altas

Estas métricas dependen de propiedades **locales** (un enemigo sobre su tile vecino,  
un gap de pocos tiles de ancho). El DCGAN las aprendió bien desde el entrenamiento  
adversarial porque están sobrerrepresentadas en los datos originales: los niveles de  
Mario casi siempre tienen enemigos sobre suelo y huecos estrechos. Por esto, CMA-ES  
las lleva a 1.0 rápidamente y concentra el presupuesto de evaluaciones en mejorar  
`pipe_completeness` y `ground_continuity`.